# Day 31 — Practice ANSWERS
### What Machine Learning Actually Is

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it asks you to do | How many |
|---|---|---|
| **A** | Work it out — no computer needed | 8 |
| **B** | Find the mistake in a short program | 4 |
| **C** | Write the code yourself | 5 |
| **D** | Challenge | 2 |

### How to work

**Part A is pen-and-paper first.** Write your answer, *then* run the cell to check.

Three questions to keep asking yourself:

- Is this **supervised** or **unsupervised**? If supervised, **number** or **group**?
- Does this column have a **real order**, or am I inventing one?
- Is this score measured on rows the model has **already seen**?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

import warnings
warnings.simplefilter("ignore")
plt.rcParams["figure.figsize"] = (8, 3.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("ready")

ready


---
# Part A — Work it out

Eight questions. Most need no computer at all.

### A1 · Read the table

Below is a small dataset of second-hand scooters. Before running anything, write down:

1. How many **samples** are there?
2. How many **features**?
3. Which column is the **label**, if we want to predict the price?

In [2]:
scooters = pd.DataFrame({
    "brand":    ["Honda", "TVS", "Bajaj", "Honda", "Hero"],
    "age_yr":   [3, 1, 5, 2, 4],
    "km_run":   [18000, 4200, 31000, 11500, 25000],
    "mileage":  [45, 52, 40, 45, 48],
    "price_k":  [52, 78, 31, 61, 42],
})
scooters

,brand,age_yr,km_run,mileage,price_k
0,Honda,3,18000,45,52
1,TVS,1,4200,52,78
2,Bajaj,5,31000,40,31
3,Honda,2,11500,45,61
4,Hero,4,25000,48,42


1. **5 samples** — one per row, one per scooter.
2. **4 features** — brand, age_yr, km_run, mileage. These are what we know.
3. The label is **price_k** — the thing we are trying to predict.

A simple way to remember it: the label is the column you would have to *pay someone* to
find out. The features are the ones you can read off the scooter yourself.

### A2 · Name the problem

For each of these, say: supervised or unsupervised? And if supervised, number or group?

| | The problem |
|---|---|
| a | Predict a scooter's resale price from its age and kilometres |
| b | Decide whether a student will pass or fail |
| c | Split 5,000 Instagram users into groups by what they post |
| d | Predict how many minutes a train will be delayed |
| e | Decide which of 10 diseases a scan shows |

| | Answer | Why |
|---|---|---|
| a | **Supervised · Regression** | We have old sale prices, and a price is a number |
| b | **Supervised · Classification** | We have past results, and the answer is one of two groups |
| c | **Unsupervised** | Nobody has sorted the users — there are no answers to learn from |
| d | **Supervised · Regression** | We have past delays, and minutes is a number |
| e | **Supervised · Classification** | We have labelled scans, and the answer is one of 10 groups |

Notice that **(e) still counts as classification even with 10 groups.** Classification is
not only yes/no — it is any problem where the answer is one of a fixed set of choices.

### A3 · Count the columns

If we one-hot encode the `brand` column in the scooter table, how many columns will the
whole table have afterwards? Work it out before running.

In [3]:
encoded = pd.get_dummies(scooters, columns=["brand"], dtype=int)
print("before:", scooters.shape[1], "columns")
print("after :", encoded.shape[1], "columns")
print()
print(list(encoded.columns))

before: 5 columns
after : 8 columns

['age_yr', 'km_run', 'mileage', 'price_k', 'brand_Bajaj', 'brand_Hero', 'brand_Honda', 'brand_TVS']


**Before: 5 columns. After: 8 columns.**

The arithmetic: we **remove** 1 column (`brand`) and **add** 4 — one for each distinct
brand: Honda, TVS, Bajaj, Hero. So 5 − 1 + 4 = **8**.

The general rule:

> new total  =  old total  −  1  +  (number of different values in that column)

Note that "Honda" appears twice but still gets only **one** column. What matters is how
many *distinct* values there are, not how many rows.

### A4 · Which of these columns has a real order?

For each one, say whether you would **one-hot encode** it or **just number it**.

| | The column | Values |
|---|---|---|
| a | `blood_group` | A+, B+, O+, AB+ |
| b | `t_shirt_size` | S, M, L, XL |
| c | `city` | Mumbai, Delhi, Chennai |
| d | `satisfaction` | Poor, Average, Good, Excellent |
| e | `has_licence` | yes, no |

| | Answer | Why |
|---|---|---|
| a | **one-hot** | O+ is not "more" than A+. They are just different. |
| b | **number them 1,2,3,4** | XL genuinely is larger than S. The order is real. |
| c | **one-hot** | Delhi is not more than Mumbai. |
| d | **number them 1,2,3,4** | Excellent genuinely is better than Poor. |
| e | **one column of 0 and 1** | With only two options you never need two columns. |

The test is always the same one sentence: **"is one of these genuinely MORE than
another?"** If you cannot answer yes with a straight face, use one-hot.

### A5 · Spot what the numbering is claiming

Someone encoded `city` as Chennai=0, Delhi=1, Mumbai=2.

Write down **three things** the model now believes that are not true.

In [4]:
cities = {"Chennai": 0, "Delhi": 1, "Mumbai": 2}
print(cities)
print()
print("the model can now do arithmetic with these. What does it conclude?")

{'Chennai': 0, 'Delhi': 1, 'Mumbai': 2}

the model can now do arithmetic with these. What does it conclude?


Three false beliefs, among many:

1. **Mumbai (2) is twice Delhi (1).**
2. **Delhi sits exactly halfway between Chennai and Mumbai.** (0 + 2) ÷ 2 = 1.
3. **Mumbai − Delhi = Delhi − Chennai**, so the "gap" between Delhi and Mumbai is the same
   size as the gap between Chennai and Delhi.

None of these means anything. Cities have no order, no midpoint and no spacing.

And here is the part that makes it dangerous: **nothing in the code warns you.** It runs
cleanly and produces a confident answer built on nonsense.

### A6 · Which score should you report?

A classmate shows you this and says the model is 94% accurate.

In [5]:
rng = np.random.default_rng(55)
x = rng.uniform(0, 10, 40)
y = 20 + 3*x + rng.normal(0, 5, 40)

X_study, X_hidden, y_study, y_hidden = train_test_split(
    x.reshape(-1, 1), y, test_size=0.3, random_state=0)

tree = DecisionTreeRegressor(random_state=0).fit(X_study, y_study)

print("score on the rows it studied :", round(tree.score(X_study, y_study), 3))
print("score on the HIDDEN rows     :", round(tree.score(X_hidden, y_hidden), 3))

score on the rows it studied : 1.0
score on the HIDDEN rows     : 0.291


```
score on the rows it studied : 1.000
score on the HIDDEN rows     : 0.291
```

**Report 0.291.** The 1.0 is worthless.

A decision tree with no limit on its size can always reach a perfect score on the rows it
studied — it simply grows one branch per row and remembers each answer. That is
**memorising**, not learning, and the proof is right there in the second number.

Whenever you see a perfect score, your first thought should be *"what went wrong?"*, not
*"how good!"*

### A7 · What does the error tell you?

Our model guesses laptop prices. Here are its five guesses next to the truth. Work out the
total squared error by hand, then check.

In [6]:
truth   = np.array([30., 45., 52., 38., 61.])
guesses = np.array([32., 42., 52., 40., 59.])

errors = truth - guesses
print("the errors        :", errors)
print("squared           :", errors ** 2)
print("added up          :", (errors ** 2).sum())

the errors        : [-2.  3.  0. -2.  2.]
squared           : [4. 9. 0. 4. 4.]
added up          : 21.0


Errors: `[-2, +3, 0, -2, +2]`. Squared: `[4, 9, 0, 4, 4]`. Total: **21.0**

Two things worth noticing:

- The **zero** means the model got that laptop exactly right. Squaring leaves it at zero,
  so a perfect guess costs nothing.
- The **+3** contributes 9 — more than the two −2s put together (4 + 4 = 8). Squaring
  makes a bigger miss cost disproportionately more, which is usually what you want.

A model that is slightly off every time is more useful than one that is perfect four times
and badly wrong once.

### A8 · How many columns is too many?

A delivery dataset has 2,000 rows and a `restaurant_name` column with 1,350 different
restaurants in it.

What happens if you one-hot encode it? Answer in words before running.

In [7]:
rows = 2000
distinct_restaurants = 1350

print("rows                      :", rows)
print("columns after one-hot     :", distinct_restaurants)
print("rows per column (average) :", round(rows / distinct_restaurants, 2))

rows                      : 2000
columns after one-hot     : 1350
rows per column (average) : 1.48


**1,350 new columns from 2,000 rows** — about 1.5 rows per column.

Most restaurants will appear **once or twice in the entire dataset**. A column that is 1
for a single row tells the model nothing general; it just lets it memorise that one row.

**What to do instead:** keep the 20 or 30 busiest restaurants as their own columns and
label everything else `"OTHER"`. You lose a little detail and you gain a model that
actually works on new data.

A rough guide: if a category has fewer than about 20–30 rows, it probably belongs in
`"OTHER"`.

---
# Part B — Find the mistake

Four short programs. Each one either crashes or gives a misleading answer. Find it, say
what is wrong in one sentence, then fix it.

### B1 · This one crashes

In [8]:
cars = pd.DataFrame({
    "fuel":   ["petrol", "diesel", "petrol", "cng"],
    "age_yr": [3, 5, 2, 7],
    "price":  [420, 310, 510, 180],
})

X = cars[["fuel", "age_yr"]]
y = cars["price"]

try:
    LinearRegression().fit(X, y)
except Exception as e:
    print(type(e).__name__, ":", e)

ValueError : could not convert string to float: 'petrol'


**The problem:** the `fuel` column contains words, and a model can only do arithmetic.
There is no answer to `"petrol" × 0.4`.

**The fix:** turn the word column into numbers first, with `pd.get_dummies`.

In [9]:
cars_ready = pd.get_dummies(cars, columns=["fuel"], dtype=int)
print(cars_ready)
print()

X = cars_ready.drop(columns=["price"])
y = cars_ready["price"]
model = LinearRegression().fit(X, y)
print("it works now. columns used:", list(X.columns))

   age_yr  price  fuel_cng  fuel_diesel  fuel_petrol
0       3    420         0            0            1
1       5    310         0            1            0
2       2    510         0            0            1
3       7    180         1            0            0

it works now. columns used: ['age_yr', 'fuel_cng', 'fuel_diesel', 'fuel_petrol']


### B2 · It runs, and the answer is wrong

In [10]:
rng = np.random.default_rng(77)
N = 400

city  = rng.choice(["Chennai", "Delhi", "Mumbai"], N)
extra = pd.Series(city).map({"Chennai": 0, "Delhi": 5, "Mumbai": 40}).values
size  = rng.integers(400, 1600, N)
rent  = 3000 + 12*size + 400*extra + rng.normal(0, 1500, N)

homes = pd.DataFrame({"city": city, "size_sqft": size, "rent": rent})

# the analyst numbers the cities alphabetically
numbering = {"Chennai": 0, "Delhi": 1, "Mumbai": 2}
bad = homes.copy()
bad["city"] = bad["city"].map(numbering)          # <-- look here

X_study, X_hidden, y_study, y_hidden = train_test_split(
    bad.drop(columns=["rent"]), bad["rent"], test_size=.3, random_state=0)
m = LinearRegression().fit(X_study, y_study)
print("average error : ₹", round(mean_absolute_error(y_hidden, m.predict(X_hidden))))

average error : ₹ 2668


**The problem:** numbering the cities 0, 1, 2 tells the model that Mumbai is *twice* Delhi
and that Delhi sits exactly halfway between Chennai and Mumbai.

In this data that is badly wrong. The real city premiums are 0, 5 and 40 — Mumbai is far
more expensive than Delhi, not merely "one step up".

**The fix:** one-hot encode the city column so no city is bigger than another.

In [11]:
good = pd.get_dummies(homes, columns=["city"], dtype=int)

for name, table in [("numbering the cities", bad), ("one-hot encoding   ", good)]:
    X_study, X_hidden, y_study, y_hidden = train_test_split(
        table.drop(columns=["rent"]), table["rent"], test_size=.3, random_state=0)
    m = LinearRegression().fit(X_study, y_study)
    err = mean_absolute_error(y_hidden, m.predict(X_hidden))
    print(f"{name}:  average error ₹{err:>7,.0f}   score {m.score(X_hidden, y_hidden):.3f}")

numbering the cities:  average error ₹  2,668   score 0.879
one-hot encoding   :  average error ₹  1,267   score 0.969


```
numbering the cities:  average error ₹  2,668   score 0.879
one-hot encoding   :  average error ₹  1,267   score 0.969
```

The average error **more than halved** — ₹2,668 down to ₹1,267 — and the score went from
0.879 to 0.969. Same data, same model, same size column. Only the encoding changed.

This is exactly the laptop-brand result from the lecture, on a different dataset. The
mistake is not rare and it is not harmless.

### B3 · A suspiciously good model

In [12]:
marks = pd.DataFrame({
    "hours":      [2, 4, 5, 7, 8, 3, 6, 9, 1, 5],
    "attendance": [60, 75, 80, 90, 95, 65, 85, 98, 50, 78],
    "final_mark": [41, 55, 62, 74, 81, 48, 68, 88, 35, 60],
})

X = marks                            # <-- look here
y = marks["final_mark"]

X_study, X_hidden, y_study, y_hidden = train_test_split(X, y, test_size=.3, random_state=0)
m = LinearRegression().fit(X_study, y_study)
print("score on hidden rows:", round(m.score(X_hidden, y_hidden), 6))

score on hidden rows: 1.0


**The problem:** `X = marks` includes **every** column — including `final_mark`, which is
the answer we are trying to predict.

So the model was handed the answer as one of its inputs. It learned the devastatingly
clever rule *"final_mark = final_mark"* and scored a perfect 1.000.

The moment you use it on a real new student — who has no final mark yet, because that is
the whole point — it has nothing to work with.

**The fix:** drop the label column from the inputs.

In [13]:
X = marks.drop(columns=["final_mark"])      # inputs only
y = marks["final_mark"]                     # the answer, kept separate

X_study, X_hidden, y_study, y_hidden = train_test_split(X, y, test_size=.3, random_state=0)
m = LinearRegression().fit(X_study, y_study)

print("inputs used         :", list(X.columns))
print("score on hidden rows:", round(m.score(X_hidden, y_hidden), 4))

inputs used         : ['hours', 'attendance']
score on hidden rows: 0.9988


Now the score is high but **not perfect**, which is what an honest model looks like on real
data.

**The habit:** always build X by dropping the label, never by listing columns you hope are
right. `df.drop(columns=["answer"])` can't accidentally include the answer. Typing out a
list of columns can.

### B4 · The score changes every time

In [14]:
rng = np.random.default_rng(99)
x = rng.uniform(0, 10, 50)
y = 10 + 2*x + rng.normal(0, 4, 50)

for attempt in range(4):
    a, b, c, d = train_test_split(x.reshape(-1,1), y, test_size=.3)   # <-- look here
    m = LinearRegression().fit(a, c)
    print(f"run {attempt+1}:  score = {m.score(b, d):.4f}")

run 1:  score = 0.7095
run 2:  score = 0.6725
run 3:  score = 0.8093
run 4:  score = 0.7367


**The problem:** `train_test_split` picks the hidden rows **at random**, and without
`random_state` it picks differently every single run.

So the score moves around — here by several hundredths between runs. You cannot compare
two models, you cannot reproduce your own result, and you cannot tell a real improvement
from luck.

**The fix:** set `random_state` to any fixed number.

In [15]:
for attempt in range(4):
    a, b, c, d = train_test_split(x.reshape(-1,1), y, test_size=.3, random_state=0)
    m = LinearRegression().fit(a, c)
    print(f"run {attempt+1}:  score = {m.score(b, d):.4f}")

print()
print("Identical every time. Now any change you see is caused by YOUR change,")
print("not by the computer shuffling the rows differently.")

run 1:  score = 0.4908
run 2:  score = 0.4908
run 3:  score = 0.4908
run 4:  score = 0.4908

Identical every time. Now any change you see is caused by YOUR change,
not by the computer shuffling the rows differently.


---
# Part C — Write the code

Five tasks on one dataset. Write your answer in the empty cell; a worked solution follows.

In [16]:
# ---------------------------------------------------------------
# 700 second-hand phones. Run this once - tasks C1 to C5 use `phones`.
# ---------------------------------------------------------------
rng = np.random.default_rng(3131)
n = 700

brand   = rng.choice(["Apple", "Samsung", "Xiaomi", "Realme"], n, p=[.22, .30, .26, .22])
premium = pd.Series(brand).map({"Apple": 38, "Samsung": 12, "Xiaomi": 2, "Realme": 0}).values
condition = rng.choice(["Poor", "Fair", "Good", "Excellent"], n)
cond_bump = pd.Series(condition).map({"Poor": 0, "Fair": 6, "Good": 12, "Excellent": 18}).values
ram     = rng.choice([4, 6, 8, 12], n)
age_mo  = rng.integers(1, 48, n)

price = (8 + 1.6*ram + premium + cond_bump - 0.22*age_mo + rng.normal(0, 3, n))

phones = pd.DataFrame({"brand": brand, "condition": condition, "ram_gb": ram,
                       "age_months": age_mo, "price_k": price.round(1)})
phones.head()

,brand,condition,ram_gb,age_months,price_k
0,Xiaomi,Poor,6,34,14.3
1,Realme,Good,4,17,21.1
2,Samsung,Excellent,8,37,40.1
3,Realme,Poor,4,39,2.9
4,Xiaomi,Good,12,47,25.8


### C1 · Describe the dataset

Print, in plain language:

1. how many samples there are
2. the names of the features
3. the name of the label (we want to predict `price_k`)
4. which columns are words and will need encoding

In [17]:
# your code here

In [18]:
print("samples :", len(phones))
print("label   : price_k")
print("features:", [c for c in phones.columns if c != "price_k"])
print()

is_number = pd.api.types.is_numeric_dtype
word_columns = [c for c in phones.columns if not is_number(phones[c])]
print("columns made of WORDS (these need encoding):", word_columns)
for c in word_columns:
    print(f"   {c:<12} has {phones[c].nunique()} different values: {sorted(phones[c].unique())}")

samples : 700
label   : price_k
features: ['brand', 'condition', 'ram_gb', 'age_months']

columns made of WORDS (these need encoding): ['brand', 'condition']
   brand        has 4 different values: ['Apple', 'Realme', 'Samsung', 'Xiaomi']
   condition    has 4 different values: ['Excellent', 'Fair', 'Good', 'Poor']


700 samples, 4 features (`brand`, `condition`, `ram_gb`, `age_months`), and `price_k` is
the label.

**Two columns are words**, and they need completely different treatment:

- `brand` — Apple, Samsung, Xiaomi, Realme. **No order.** Apple is not "more" than Xiaomi.
  → one-hot
- `condition` — Poor, Fair, Good, Excellent. **Real order.** Excellent genuinely is better
  than Poor. → number them 1, 2, 3, 4

Spotting that difference is the whole point of this task, and it is a judgement only you
can make. `pandas` cannot tell you whether an order is real.

### C2 · Encode the two word columns — each the right way

Build a new table where:

- `brand` is one-hot encoded
- `condition` is numbered 1 (Poor) to 4 (Excellent)

Print how many columns you end up with.

In [19]:
# your code here

In [20]:
ready = phones.copy()

# condition HAS a real order -> number it
condition_order = {"Poor": 1, "Fair": 2, "Good": 3, "Excellent": 4}
ready["condition"] = ready["condition"].map(condition_order)

# brand has NO order -> one-hot
ready = pd.get_dummies(ready, columns=["brand"], dtype=int)

print("before:", phones.shape[1], "columns ->", list(phones.columns))
print("after :", ready.shape[1], "columns ->", list(ready.columns))
ready.head(3)

before: 5 columns -> ['brand', 'condition', 'ram_gb', 'age_months', 'price_k']
after : 8 columns -> ['condition', 'ram_gb', 'age_months', 'price_k', 'brand_Apple', 'brand_Realme', 'brand_Samsung', 'brand_Xiaomi']


,condition,ram_gb,age_months,price_k,brand_Apple,brand_Realme,brand_Samsung,brand_Xiaomi
0,1,6,34,14.3,0,0,0,1
1,3,4,17,21.1,0,1,0,0
2,4,8,37,40.1,0,0,1,0


**5 columns become 8.**

The arithmetic: we started with 5. `condition` stays as 1 column (just renumbered).
`brand` becomes 4 columns, replacing 1. So 5 − 1 + 4 = **8**.

Had we one-hot encoded `condition` too, we would have had 11 columns for no benefit — and
we would have thrown away the useful fact that Excellent > Good > Fair > Poor.

### C3 · Train and score it honestly

Hide 30% of the rows (`random_state=0`), train a `LinearRegression`, and report:

- the score on the hidden rows
- the score on the studied rows
- the average error in thousands of rupees

In [21]:
# your code here

In [22]:
X = ready.drop(columns=["price_k"])
y = ready["price_k"]

X_study, X_hidden, y_study, y_hidden = train_test_split(X, y, test_size=0.3, random_state=0)
model = LinearRegression().fit(X_study, y_study)

print(f"score on studied rows : {model.score(X_study, y_study):.4f}")
print(f"score on HIDDEN rows  : {model.score(X_hidden, y_hidden):.4f}")
print(f"average error         : ₹{mean_absolute_error(y_hidden, model.predict(X_hidden)):.2f}k")

score on studied rows : 0.9682
score on HIDDEN rows  : 0.9677
average error         : ₹2.47k


```
score on studied rows : 0.9682
score on HIDDEN rows  : 0.9677
average error         : ₹2.47k
```

Two things worth noticing:

**The two scores are almost identical.** That is a healthy sign. A big gap between them is
the warning sign of memorising — and a straight line with 7 inputs and 490 training rows
simply does not have enough flexibility to memorise anything.

**The error is close to the noise floor.** Look back at how the data was built: we added
`rng.normal(0, 3, n)` — about ₹3k of pure randomness per phone. No model could ever beat
that, and we are at ₹2.47k. We are already at the limit of what this data allows.

### C4 · Look inside the model

Print each input column next to the weight the model learned for it.

Then answer in a comment: **which brand is worth the most**, and does the model agree with
how the data was built?

In [23]:
# your code here

In [24]:
weights = pd.DataFrame({
    "column": X.columns,
    "weight": model.coef_.round(2),
}).sort_values("weight", ascending=False)

print(weights.to_string(index=False))
print(f"\n(starting point: {model.intercept_:.2f})")

       column  weight
  brand_Apple   25.43
    condition    5.97
       ram_gb    1.69
   age_months   -0.20
brand_Samsung   -1.14
 brand_Xiaomi  -11.38
 brand_Realme  -12.91

(starting point: 13.92)


The brand weights line up in exactly the order we built in: **Apple highest, then Samsung,
then Xiaomi, then Realme last.** The gaps roughly match the premiums we used (38, 12, 2, 0).

The other weights also make sense:

- `ram_gb` comes out at **+1.69** — we built it with 1.6.
- `age_months` comes out at **−0.20** — negative, because older phones are worth less.
  We built it with −0.22.
- `condition` comes out at **+5.97**, because each step up the 1→4 ladder adds about ₹6k.

(The brand weights look shifted — Apple +25 rather than +38, and the others negative —
because the starting point of ₹13.92k has absorbed part of the overall level. What matters
is the **gaps between them**, and those are right.)

**The model was never told any of this.** It found all of it from 490 rows of prices. That
is what makes the whole exercise worth doing.

### C5 · Prove the encoding mattered

Build a **second** version where `brand` is numbered alphabetically instead of one-hot
encoded, and compare the two on hidden rows.

In [25]:
# your code here

In [26]:
# the wrong way - number the brands alphabetically
wrong = phones.copy()
wrong["condition"] = wrong["condition"].map(condition_order)
alphabetical = {b: i for i, b in enumerate(sorted(phones.brand.unique()))}
wrong["brand"] = wrong["brand"].map(alphabetical)
print("the numbering it invents:", alphabetical)
print()

for name, table in [("numbering the brands", wrong), ("one-hot encoding   ", ready)]:
    X_ = table.drop(columns=["price_k"])
    y_ = table["price_k"]
    a, b, c, d = train_test_split(X_, y_, test_size=0.3, random_state=0)
    m = LinearRegression().fit(a, c)
    print(f"{name}:  average error ₹{mean_absolute_error(d, m.predict(b)):5.2f}k   "
          f"score {m.score(b, d):.4f}")

the numbering it invents: {'Apple': 0, 'Realme': 1, 'Samsung': 2, 'Xiaomi': 3}

numbering the brands:  average error ₹ 8.36k   score 0.6312


one-hot encoding   :  average error ₹ 2.47k   score 0.9677


```
the numbering it invents: {'Apple': 0, 'Realme': 1, 'Samsung': 2, 'Xiaomi': 3}

numbering the brands:  average error ₹ 8.36k   score 0.6312
one-hot encoding   :  average error ₹ 2.47k   score 0.9677
```

The error is **more than three times larger**, and the score falls from 0.97 to 0.63.

And look at *why* it is so bad here. Alphabetical order puts **Apple at 0 and Xiaomi at 3**
— but Apple is the most expensive brand and Xiaomi is nearly the cheapest. The numbering is
not merely meaningless, it is almost exactly **backwards**.

That is worth sitting with. The damage from this mistake depends entirely on how unlucky
the alphabet happens to be, which means **you cannot predict how bad it will be** — only
that you should never do it.

---
# Part D — Challenge

Two tasks. Both are marked.

### D1 · Make a broken dataset trainable

The table below cannot be fed to any model as it stands. There are **four** separate
problems with it.

**Your tasks:**

1. Find all four. Write each one down before you fix anything.
2. Fix them, in a sensible order.
3. Train a model and report an honest score.
4. Say in a comment which fix you think mattered most, and why.

In [27]:
rng = np.random.default_rng(4141)
n = 500

area   = rng.integers(500, 2500, n)
loc    = rng.choice(["Andheri", "Bandra", "Dadar", "Thane", "Powai"], n)
loc_b  = pd.Series(loc).map({"Andheri": 8, "Bandra": 30, "Dadar": 14,
                             "Thane": 0, "Powai": 18}).values
furn   = rng.choice(["Unfurnished", "Semi", "Full"], n)
furn_b = pd.Series(furn).map({"Unfurnished": 0, "Semi": 7, "Full": 15}).values
pin    = rng.integers(400001, 400001 + 300, n).astype(str)

rent = 5000 + 22*area + 900*loc_b + 700*furn_b + rng.normal(0, 2500, n)

flats = pd.DataFrame({
    "area_sqft":   area,
    "locality":    loc,
    "furnishing":  furn,
    "pincode":     pin,
    "rent":        rent.round(0),
    "rent_in_usd": (rent / 83).round(2),       # the same rent, in dollars
})
flats.head()

,area_sqft,locality,furnishing,pincode,rent,rent_in_usd
0,706,Powai,Semi,400083,40709.0,490.47
1,1078,Thane,Unfurnished,400088,29674.0,357.51
2,1921,Powai,Semi,400019,63921.0,770.13
3,2040,Dadar,Semi,400183,66827.0,805.14
4,1733,Bandra,Full,400241,82546.0,994.53


In [28]:
# your investigation here

In [29]:
# ---- PROBLEM 1: three columns are words and cannot be multiplied ----
is_number = pd.api.types.is_numeric_dtype
print("columns made of words:", [c for c in flats.columns if not is_number(flats[c])])

# ---- PROBLEM 2: pincode has far too many different values ----
print("different pincodes   :", flats.pincode.nunique(), "from", len(flats), "rows")

# ---- PROBLEM 3: furnishing HAS a real order, so one-hot would waste it ----
print("furnishing values    :", sorted(flats.furnishing.unique()))

# ---- PROBLEM 4: rent_in_usd IS the answer, just divided by 83 ----
ratio = flats.rent / flats.rent_in_usd
print(f"rent / rent_in_usd   : always {ratio.min():.1f} to {ratio.max():.1f}  <-- it IS the answer")

columns made of words: ['locality', 'furnishing', 'pincode']
different pincodes   : 248 from 500 rows
furnishing values    : ['Full', 'Semi', 'Unfurnished']
rent / rent_in_usd   : always 83.0 to 83.0  <-- it IS the answer


In [30]:
clean = flats.copy()

# FIX 4 first - remove the column that is secretly the answer
clean = clean.drop(columns=["rent_in_usd"])

# FIX 2 - group the rare pincodes before encoding anything
top = clean.pincode.value_counts().head(8).index
clean["pincode"] = clean.pincode.where(clean.pincode.isin(top), "OTHER")
print("pincodes after grouping:", clean.pincode.nunique())

# FIX 3 - furnishing has a real order, so number it
clean["furnishing"] = clean["furnishing"].map({"Unfurnished": 1, "Semi": 2, "Full": 3})

# FIX 1 - one-hot the two columns that have no order
clean = pd.get_dummies(clean, columns=["locality", "pincode"], dtype=int)

print("final shape:", clean.shape)

pincodes after grouping: 9
final shape: (500, 17)


In [31]:
X = clean.drop(columns=["rent"])
y = clean["rent"]

X_study, X_hidden, y_study, y_hidden = train_test_split(X, y, test_size=0.3, random_state=0)
model = LinearRegression().fit(X_study, y_study)

print(f"score on studied rows : {model.score(X_study, y_study):.4f}")
print(f"score on HIDDEN rows  : {model.score(X_hidden, y_hidden):.4f}")
print(f"average error         : ₹{mean_absolute_error(y_hidden, model.predict(X_hidden)):,.0f}")
print(f"\n(we built in ₹2,500 of random noise per flat, so this is about the best possible)")

score on studied rows : 0.9758
score on HIDDEN rows  : 0.9749
average error         : ₹2,017

(we built in ₹2,500 of random noise per flat, so this is about the best possible)


In [32]:
# What would have happened WITHOUT each fix?
print("if we had kept rent_in_usd:")
leak = pd.get_dummies(flats.drop(columns=["locality", "pincode"]),
                      columns=["furnishing"], dtype=int)
a, b, c, d = train_test_split(leak.drop(columns=["rent"]), leak["rent"],
                              test_size=.3, random_state=0)
print(f"   score = {LinearRegression().fit(a, c).score(b, d):.6f}   <-- a perfect model that knows nothing")

print("\nif we had one-hot encoded all 300 pincodes:")
wide = flats.drop(columns=["rent_in_usd"])
wide = pd.get_dummies(wide, columns=["locality", "furnishing", "pincode"], dtype=int)
a, b, c, d = train_test_split(wide.drop(columns=["rent"]), wide["rent"],
                              test_size=.3, random_state=0)
m = LinearRegression().fit(a, c)
print(f"   columns = {a.shape[1]}")
print(f"   studied rows {m.score(a, c):.4f}   HIDDEN rows {m.score(b, d):.4f}")

if we had kept rent_in_usd:
   score = 1.000000   <-- a perfect model that knows nothing

if we had one-hot encoded all 300 pincodes:


   columns = 257
   studied rows 0.9902   HIDDEN rows 0.9548


### The four problems

| # | Problem | Fix |
|---|---|---|
| 1 | `locality`, `furnishing` and `pincode` are **words** | encode them |
| 2 | `pincode` has **248 different values** from 500 rows | group the rare ones into "OTHER" |
| 3 | `furnishing` has a **real order** (Unfurnished < Semi < Full) | number it 1, 2, 3 — not one-hot |
| 4 | `rent_in_usd` **is the answer**, divided by 83 | drop it entirely |

### Which fix mattered most?

**Number 4, by a wide margin — and it is the one with no error message.**

The other three announce themselves. Leave the words in and the code crashes. One-hot 259
pincodes and you can count the columns. But `rent_in_usd` runs perfectly and hands you a
**score of 1.000000**, which looks like triumph.

It is not. The model learned `rent = 83 × rent_in_usd`, which is arithmetic, not
prediction. The moment you use it on a flat whose rent you do not know — which is the only
time you would ever use it — the column is empty and the model has nothing.

**The test that catches it:** for every column, ask *"would I actually know this value at
the moment I need the prediction?"* If the answer is no, the column cannot be used, however
helpful it looks.

### The honest result

After all four fixes: **0.9749 on hidden rows**, with an average error of **₹2,017**
against the ₹2,500 of noise we built in. Real, useful, and as good as this data allows.

And the pincode comparison shows the second trap clearly:

```
grouped pincodes (17 columns) : studied 0.9758   HIDDEN 0.9749
all pincodes    (257 columns) : studied 0.9902   HIDDEN 0.9548
```

Keeping every pincode gives a **better score on studied rows** and a **worse one on hidden
rows**. That gap opening up is the signature of a model that has started memorising.

### D2 · Write a reusable checker

Write a function `check_dataset(df, label)` that looks at any table and prints a short
report before anyone trains anything on it.

It must report:

1. how many samples and how many features
2. which columns are words, and how many different values each has
3. a **warning** for any word column with more than 20 different values
4. a **warning** for any column that is suspiciously close to a copy of the label
5. a suggested encoding for each word column, based on how many values it has

Then run it on `flats` and show that it catches all four problems from D1.

In [33]:
# your code here

In [34]:
def check_dataset(df, label):
    """Print a short health report on a table, before anyone trains a model on it."""

    print("=" * 62)
    print(f"  DATASET CHECK   ({len(df)} samples, {df.shape[1]-1} features)")
    print(f"  predicting: {label}")
    print("=" * 62)

    warnings_found = 0

    # ---- 1. look at every column that is NOT the label ----
    for col in df.columns:
        if col == label:
            continue

        # ---- words, or numbers? ----
        if not pd.api.types.is_numeric_dtype(df[col]):
            k = df[col].nunique()
            if k > 20:
                print(f"  [!] {col:<14} WORDS, {k} different values")
                print(f"      -> too many. Keep the top 10-20 and group the rest as 'OTHER'.")
                warnings_found += 1
            elif k == 2:
                print(f"  [ ] {col:<14} WORDS, 2 values -> one column of 0 and 1")
            else:
                print(f"  [ ] {col:<14} WORDS, {k} values -> one-hot, unless they have a real order")
        else:
            # ---- is this column secretly the label? ----
            r = abs(df[col].corr(df[label]))
            if r > 0.999:
                print(f"  [!] {col:<14} NUMBER, but it tracks {label} almost exactly (r = {r:.5f})")
                print(f"      -> is this the answer in disguise? If so, DROP IT.")
                warnings_found += 1
            else:
                print(f"  [ ] {col:<14} NUMBER, ready to use")

    print("-" * 62)
    print(f"  {warnings_found} warning(s). Fix these before training.")
    print("=" * 62)
    return warnings_found

In [35]:
_ = check_dataset(flats, label="rent")

  DATASET CHECK   (500 samples, 5 features)
  predicting: rent
  [ ] area_sqft      NUMBER, ready to use
  [ ] locality       WORDS, 5 values -> one-hot, unless they have a real order
  [ ] furnishing     WORDS, 3 values -> one-hot, unless they have a real order
  [!] pincode        WORDS, 248 different values
      -> too many. Keep the top 10-20 and group the rest as 'OTHER'.
  [!] rent_in_usd    NUMBER, but it tracks rent almost exactly (r = 1.00000)
      -> is this the answer in disguise? If so, DROP IT.
--------------------------------------------------------------
  2 warning(s). Fix these before training.


In [36]:
# And on the cleaned-up version, to show the warnings clear
print()
tidy = flats.drop(columns=["rent_in_usd"]).copy()
top = tidy.pincode.value_counts().head(8).index
tidy["pincode"] = tidy.pincode.where(tidy.pincode.isin(top), "OTHER")
_ = check_dataset(tidy, label="rent")


  DATASET CHECK   (500 samples, 4 features)
  predicting: rent


  [ ] area_sqft      NUMBER, ready to use
  [ ] locality       WORDS, 5 values -> one-hot, unless they have a real order
  [ ] furnishing     WORDS, 3 values -> one-hot, unless they have a real order
  [ ] pincode        WORDS, 9 values -> one-hot, unless they have a real order
--------------------------------------------------------------
  0 warning(s). Fix these before training.


### What the checker catches

On the raw `flats` table it raises **two warnings**:

- `pincode` — hundreds of different values, far too many to one-hot
- `rent_in_usd` — tracks `rent` with a correlation of **1.00000**, which is the
  fingerprint of the answer in disguise

On the cleaned table it raises **zero**.

### Why the two things it does NOT warn about still matter

Notice what the checker reports for `locality` and `furnishing`: both are flagged as
*"one-hot, unless they have a real order"*. It cannot be more specific than that, and it
never will be able to.

**No program can tell you whether an order is real.** `Unfurnished < Semi < Full` is a real
order; `Andheri < Bandra < Dadar` is not. The only difference between them is what the
words *mean*, and that is knowledge about flats, not about data.

This is the honest boundary of automated checking: it can catch shapes and numbers, and it
can never replace knowing what your columns are.

### One thing to be careful of

The `r > 0.999` test only catches a column that is a **straight multiple** of the label. A
leaking column that is related in a messier way — say, the deposit, which is usually but
not always two months' rent — would slip through with a correlation of 0.95 and look like
a great feature.

A checker is a safety net, not a substitute for thinking about where each column came from.

---
## Before you hand it in

- Did you **encode every word column** before training?
- For each one: does it have a **real order**, or did you invent one?
- Did you **drop the label** from the inputs?
- Is your score measured on rows the model has **never seen**?
- Did you set `random_state` so your result repeats?
- If any score looked **perfect**, did you go and find out why?

Three numbers worth carrying away:

- Numbering brands instead of one-hot encoding them **tripled the error**: ₹3.3k → ₹10.2k.
- A tree scored **1.000 on rows it studied and 0.644 on new ones**. Only the second is real.
- One-hot encoding 426 pincodes from 900 rows gave a score of **−0.494** — worse than
  guessing.

Answers will be shared after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*